BTC 5-minute market / price data 

notebook builds the v0 dataset from markets and prices 

df_prices_window: every price observation from the 5-minute window start through the prediction time (many rows per market) 
df_final: latest available price at or before the prediction time (one row per market)

(note: ticks and spots prices are downloaded for future work but not used in v0)

In [1]:
from pathlib import Path
import os

import polars as pl
from datasets import load_dataset

os.environ["HF_HUB_DISABLE_SYMLINKS"] = "1"

c:\Users\Anigma PC\miniconda3\envs\btc_research\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#Load Data from HF 

markets = load_dataset(
    "aliplayer1/polymarket-crypto-updown",
    "markets")
prices = load_dataset(
    "aliplayer1/polymarket-crypto-updown",
    "prices")

ticks = load_dataset(
    "aliplayer1/polymarket-crypto-updown",
    "ticks")

spot_prices = load_dataset(
    "aliplayer1/polymarket-crypto-updown",
    "spot_prices"
)

"""
orderbook = load_dataset(
    "aliplayer1/polymarket-crypto-updown",
    "orderbook",
    streaming=True 
)"""

datasets = { 
    "../data/raw/markets_raw.parquet": markets,
    "../data/raw/prices_raw.parquet": prices,
    "../data/raw/ticks_raw.parquet": ticks,
    "../data/raw/spot_prices_raw.parquet": spot_prices
}

for filepath, dataset in datasets.items():
    if not os.path.exists(filepath):
        print(f"File missing. Processing and saving: {filepath}")
        df = dataset['train'].to_pandas()
        df.to_parquet(filepath, index=False)
    else:
        print(f"File already exists, skipping: {filepath}")


Generating train split:  10%|▉         | 15002792/150264409 [00:40<52:42, 42777.09 examples/s]Exception ignored in: <generator object ArrowBasedBuilder._prepare_split_single at 0x000001B6749D5380>
Traceback (most recent call last):
  File "c:\Users\Anigma PC\miniconda3\envs\btc_research\Lib\site-packages\datasets\builder.py", line 1745, in _prepare_split
    pbar.update(content)
RuntimeError: generator ignored GeneratorExit
Generating train split:  10%|▉         | 15002792/150264409 [00:40<06:06, 369045.96 examples/s]


PermissionError: [WinError 32] The process cannot access the file because it is being used by another process: 'C:/Users/Anigma PC/.cache/huggingface/datasets/aliplayer1___polymarket-crypto-updown/ticks/0.0.0/ffb15ce5c9bdea2c98f3503973807ec4387c668e.incomplete\\polymarket-crypto-updown-train-00000-00006-of-NNNNN.arrow'

In [3]:
df_markets = pl.scan_parquet("../data/raw/markets_raw.parquet")
df_prices = pl.scan_parquet("../data/raw/prices_raw.parquet")

In [ ]:

## NOTE: start_ts represents contract launch not 5m window ts (window_start_ts) - kept as metadata for now 

WINDOW_SECONDS = 300
PREDICTION_HORIZON_SECONDS = 60

market_columns = [
    "market_id",
    "question",
    "crypto",
    "timeframe",
    "resolution",
    "volume",
    "start_ts",
    "end_ts",
    "slug",
    "fee_rate_bps",
]

df_markets_btc5m = (
    df_markets
    .filter(
        (pl.col("crypto") == "BTC")
        & (pl.col("timeframe") == "5-minute")
    )
    .select(market_columns)
    .with_columns(
        (pl.col("end_ts") - WINDOW_SECONDS).alias("window_start_ts"),
        (pl.col("end_ts") - PREDICTION_HORIZON_SECONDS).alias("prediction_ts"),
    )
)

In [ ]:

## many rows per market (window_start_ts < p_t < prediction_ts )
## exclucdes future prices > prediction_ts to prevent data leakage 

df_prices_window = (
    df_prices
    .join(
        df_markets_btc5m.select(
            "market_id",
            "window_start_ts",
            "prediction_ts",
        ),
        on="market_id",
        how="inner",
    )
    .filter(
        pl.col("timestamp").is_between(
            pl.col("window_start_ts"),
            pl.col("prediction_ts"),
            closed="both",
        )
    )
    .select(
        "market_id",
        "timestamp",
        "up_price",
        "down_price",
        "window_start_ts",
        "prediction_ts",
    )
    .sort(["market_id", "timestamp"])
    .collect()
)

df_prices_window.head()

market_id,timestamp,up_price,down_price,window_start_ts,prediction_ts
str,i64,f32,f32,i64,i64
"""1367600""",1770859515,0.5,0.5,1770859500,1770859740
"""1367600""",1770859573,0.5,0.5,1770859500,1770859740
"""1367600""",1770859574,0.5,0.5,1770859500,1770859740
"""1367600""",1770859634,0.75,0.25,1770859500,1770859740
"""1367600""",1770859693,0.875,0.125,1770859500,1770859740


In [10]:
markets_for_join = df_markets_btc5m.sort(["market_id", "prediction_ts"])
prices_for_join = (
    df_prices_window.lazy()
    .rename({"timestamp": "price_timestamp"})
    .select("market_id", "price_timestamp", "up_price", "down_price")
    .sort(["market_id", "price_timestamp"])
)

df_final = (
    markets_for_join
    .join_asof(
        prices_for_join,
        left_on="prediction_ts",
        right_on="price_timestamp",
        by="market_id",
        strategy="backward",
    )
    .with_columns(
        (pl.col("prediction_ts") - pl.col("price_timestamp"))
        .alias("price_age_seconds")
    )
    .select(
        "market_id",
        "question",
        "crypto",
        "timeframe",
        "resolution",
        "volume",
        "start_ts",
        "end_ts",
        "window_start_ts",
        "prediction_ts",
        "slug",
        "fee_rate_bps",
        "price_timestamp",
        "price_age_seconds",
        "up_price",
        "down_price",
    )
    .sort("window_start_ts")
    .collect()
)

df_final.head()

C:\Users\Anigma PC\AppData\Local\Temp\ipykernel_24852\2335257649.py:41: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  .collect()


market_id,question,crypto,timeframe,resolution,volume,start_ts,end_ts,window_start_ts,prediction_ts,slug,fee_rate_bps,price_timestamp,price_age_seconds,up_price,down_price
str,str,str,str,i8,f32,i64,i64,i64,i64,str,i16,i64,i64,f32,f32
"""1367602""","""Bitcoin Up or Down - February …","""BTC""","""5-minute""",-1,298.866302,1770857030,1770859200,1770858900,1770859140,"""""",-1,1770859098,42,0.705,0.295
"""1367603""","""Bitcoin Up or Down - February …","""BTC""","""5-minute""",-1,12024.517578,1770857034,1770859500,1770859200,1770859440,"""""",-1,1770859406,34,0.5,0.5
"""1367600""","""Bitcoin Up or Down - February …","""BTC""","""5-minute""",-1,2744.22876,1770857030,1770859800,1770859500,1770859740,"""""",-1,1770859693,47,0.875,0.125
"""1367601""","""Bitcoin Up or Down - February …","""BTC""","""5-minute""",-1,739.589783,1770857036,1770860100,1770859800,1770860040,"""""",-1,1770859993,47,0.65,0.35
"""1367604""","""Bitcoin Up or Down - February …","""BTC""","""5-minute""",-1,8765.996094,1770857032,1770860400,1770860100,1770860340,"""""",-1,1770860295,45,0.9,0.1


In [12]:
%store df_final

Stored 'df_final' (DataFrame)


In [ ]:
## Quick Diagnostics 

In [11]:
market_count = df_markets_btc5m.select(pl.len()).collect().item()
unique_market_count = df_final["market_id"].n_unique()

assert df_final.height == market_count
assert unique_market_count == market_count
assert df_final.filter(
    pl.col("price_timestamp") > pl.col("prediction_ts")
).is_empty()

print(f"BTC 5-minute markets: {market_count:,}")
print(f"Markets with a prediction-time price: {df_final['up_price'].is_not_null().sum():,}")
print(f"Markets missing a prediction-time price: {df_final['up_price'].is_null().sum():,}")
print(f"Price observations in df_prices_window: {df_prices_window.height:,}")

df_final["price_age_seconds"].describe()

BTC 5-minute markets: 14,119
Markets with a prediction-time price: 14,098
Markets missing a prediction-time price: 21
Price observations in df_prices_window: 836,185


statistic,value
str,f64
"""count""",14098.0
"""null_count""",21.0
"""mean""",27.538232
"""std""",21.51254
"""min""",0.0
"""25%""",12.0
"""50%""",33.0
"""75%""",39.0
"""max""",238.0


note: dont overwrite df_final during cleaning. for modelling -> create a separate table containing resolved labels (0/1) and an explicit price-freshness threshold chosen during EDA. 

In [ ]:
# Example for the next notebook; choose the threshold during EDA.
# df_model = df_final.filter(
#     pl.col("resolution").is_in([0, 1])
#     & pl.col("up_price").is_not_null()
#     & (pl.col("price_age_seconds") <= MAX_PRICE_AGE_SECONDS)
# )

TWAP 60s FUNCTION (FOR FUTURE USE...)

In [ ]:
def build_twap_60s(markets, spot_prices):
    """Calculate a step-function spot TWAP from prediction_ts to end_ts."""
    markets_sorted = markets.sort("prediction_ts")
    spot_sorted = spot_prices.select(
        "spot_price_timestamp",
        "spot_price",
    ).sort("spot_price_timestamp")

    carry_in = (
        markets_sorted
        .join_asof(
            spot_sorted,
            left_on="prediction_ts",
            right_on="spot_price_timestamp",
            strategy="backward",
        )
        .select(
            "market_id",
            pl.col("prediction_ts").alias("spot_price_timestamp"),
            "spot_price",
        )
        .filter(pl.col("spot_price").is_not_null())
    )

    in_window = (
        markets.join_where(
            spot_sorted,
            pl.col("spot_price_timestamp") > pl.col("prediction_ts"),
            pl.col("spot_price_timestamp") <= pl.col("end_ts"),
        )
        .select("market_id", "spot_price_timestamp", "spot_price")
    )

    spot_effective = (
        pl.concat([carry_in, in_window])
        .sort(["market_id", "spot_price_timestamp"])
        .join(markets.select("market_id", "end_ts"), on="market_id")
        .with_columns(
            pl.col("spot_price_timestamp")
            .shift(-1)
            .over("market_id")
            .alias("next_spot_price_timestamp")
        )
        .with_columns(
            pl.when(pl.col("next_spot_price_timestamp").is_not_null())
            .then(
                pl.col("next_spot_price_timestamp")
                - pl.col("spot_price_timestamp")
            )
            .otherwise(pl.col("end_ts") - pl.col("spot_price_timestamp"))
            .alias("tick_weight_s")
        )
    )

    return (
        spot_effective
        .group_by("market_id")
        .agg(
            (
                (pl.col("spot_price") * pl.col("tick_weight_s")).sum()
                / pl.col("tick_weight_s").sum()
            ).alias("twap_60s"),
            pl.len().alias("twap_tick_count"),
        )
    )